# Week-1 Gate Experiments — Kaggle Runbook Notebook
See `docs/week1_gates_runbook.md` for the decision rules. **Set `RUN_SESSION` in the config cell, then Run All.**

- **SESSION A** (~8-9h): pre-flight smoke + Gate D (dumb-pooling, 3 seeds) + Gate U seed 0 + Gates R/N (eval-only)
- **SESSION B** (~10h): Gate U seeds 1-2 + Gate U-init (diag-init control, 3 seeds) + diagnostics on all arms

**SECURITY:** this notebook uses Kaggle Secrets (`Add-ons → Secrets`): add `GITHUB_TOKEN` (a *fine-grained, repo-scoped* PAT) and optionally `WANDB_KEY`. Do NOT paste raw tokens into cells — the previous session notebook had a live PAT and wandb key inline; **revoke both at github.com/settings/tokens and wandb.ai/settings**.

In [ ]:
# Cell 0: Config — set the session you are running
RUN_SESSION = "A"   # "A" or "B"

import os, glob, subprocess, shutil
CONTRASTIVE_CKPT = "/kaggle/input/models/udaheju/contrastive-best/pytorch/default/1"

def run(args, name):
    print(f"\n{'='*60}\n  {name}\n{'='*60}\n", flush=True)
    r = subprocess.run(args)
    print(("  OK: " if r.returncode == 0 else f"  WARNING (exit {r.returncode}): ") + name, flush=True)
    return r.returncode

In [ ]:
# Cell 1: Fresh clone
# OPTION 1 (paste-and-run): paste your GitHub token below WHEN RUNNING ON KAGGLE.
#   Leave it "" in the repo copy — pushing a notebook containing a live PAT to
#   GitHub triggers secret scanning, which AUTO-REVOKES the token.
# OPTION 2 (set-and-forget): add the token once as a Kaggle Secret named
#   GITHUB_TOKEN (Add-ons -> Secrets) and it works in every notebook with no pasting.
GITHUB_TOKEN = ""   # <-- paste here on Kaggle, or use Secrets

!rm -rf /kaggle/working/cvae-quadratreeMRF
token = GITHUB_TOKEN
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("No inline token and no GITHUB_TOKEN secret - trying public clone")
url = (f"https://{token}@github.com/realjules/cvae-quadratreeMRF.git"
       if token else "https://github.com/realjules/cvae-quadratreeMRF.git")
get_ipython().system(f"git clone --depth 1 {url} /kaggle/working/cvae-quadratreeMRF")
os.chdir("/kaggle/working/cvae-quadratreeMRF")
# The gate experiments REQUIRE the June-2026 gate code. Verify it's present:
assert os.path.exists("test_week1_smoke.py"), "Gate code missing - did you push the week-1 commits?"
assert "unconstrained_pairwise" in open("complete_training.py").read(), "Old complete_training.py - push the week-1 commits first"
print("Gate code present.")

In [ ]:
# Cell 2: Data paths (same as Session 1)
os.makedirs("./input/top", exist_ok=True)
os.makedirs("./input/gt", exist_ok=True)
os.makedirs("./input/gt_eroded", exist_ok=True)
!ln -sf /kaggle/input/potsdamvaihingen/ISPRS_semantic_labeling_Vaihingen/top/*.tif ./input/top/
!ln -sf /kaggle/input/potsdamvaihingen/ISPRS_semantic_labeling_Vaihingen/gts_for_participants/*.tif ./input/gt/

In [ ]:
# Cell 3 (optional): wandb - paste key below on Kaggle, or use a WANDB_KEY secret
WANDB_KEY = ""   # <-- paste here on Kaggle if you prefer; leave "" in the repo copy

key = WANDB_KEY
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_KEY")
    except Exception:
        pass
if key:
    get_ipython().system("pip install -q wandb")
    get_ipython().system(f"wandb login {key}")
else:
    print("No wandb key - running without wandb (fine for the gates)")

In [ ]:
# Cell 4: STEP 0 - back up the contrastive checkpoint (single point of failure).
# DOWNLOAD contrastive_best_BACKUP.pth from the output and store it in Drive + locally.
src = glob.glob(os.path.join(CONTRASTIVE_CKPT, "*.pth"))
assert src, f"No .pth under {CONTRASTIVE_CKPT}"
shutil.copy2(src[0], "/kaggle/working/contrastive_best_BACKUP.pth")
print("Backed up:", src[0], "->", "/kaggle/working/contrastive_best_BACKUP.pth")

In [ ]:
# Cell 5: Pre-flight smoke test (~5 min) - verifies every gate code path before burning GPU-hours
rc = run(["python", "test_week1_smoke.py"], "PRE-FLIGHT SMOKE TEST")
assert rc == 0, "Smoke test failed - fix before training"
print("Pre-flight OK")

## SESSION A — Gate D (dumb pooling ×3) + Gate U seed 0 + Gates R/N

In [ ]:
# Cell 6 [SESSION A]: GATE D - dumb-pooling receptive-field baseline, 3 seeds (~6h)
if RUN_SESSION == "A":
    for seed in [0, 1, 2]:
        run(["python", "complete_training.py",
             "--contrastive_ckpt", CONTRASTIVE_CKPT,
             "--output_dir", f"/kaggle/working/output_pct10_dumbpool_seed{seed}",
             "--labeled_percent", "10", "--epochs_seg", "30",
             "--seed", str(seed), "--dumb_pooling",
             "--data_dir", "./input"], f"GATE D: dumb pooling, seed {seed}")

In [ ]:
# Cell 7 [SESSION A]: GATE U - unconstrained reproduction, seed 0 (~2h)
if RUN_SESSION == "A":
    run(["python", "complete_training.py",
         "--contrastive_ckpt", CONTRASTIVE_CKPT,
         "--output_dir", "/kaggle/working/output_pct10_unconstrained_seed0",
         "--labeled_percent", "10", "--epochs_seg", "30",
         "--seed", "0", "--unconstrained_pairwise",
         "--data_dir", "./input"], "GATE U: unconstrained, seed 0")

In [ ]:
# Cell 8 [SESSION A]: GATES R + N - eval-only on the EXISTING Session-1 checkpoints (in the cloned repo) (~1-2h)
if RUN_SESSION == "A":
    os.makedirs("/kaggle/working/gates", exist_ok=True)
    bp = [f"eval/all_results/output_pct10_seed{s}_best.pth" for s in [0, 1, 2]]
    nobp = [f"eval/all_results/output_pct10_nobp_seed{s}_best.pth" for s in [0, 1, 2]]
    run(["python", "measure_eval_noise.py", "--seg_ckpt", bp[0], "--k", "10",
         "--output_json", "/kaggle/working/gates/eval_noise_floor.json"], "GATE N: eval-noise floor (k=10)")
    run(["python", "measure_eval_noise.py", "--seg_ckpt", *bp, "--k", "3", "--both",
         "--output_json", "/kaggle/working/gates/removal_gap_bp.json"], "GATE R: removal gap, BP checkpoints")
    run(["python", "measure_eval_noise.py", "--seg_ckpt", *nobp, "--k", "3", "--unary_only",
         "--output_json", "/kaggle/working/gates/removal_gap_nobp.json"], "GATE R: no-BP baselines")

## SESSION B — Gate U seeds 1–2 + Gate U-init (diag-init control ×3)

In [ ]:
# Cell 9 [SESSION B]: GATE U - unconstrained seeds 1, 2 (~4h)
if RUN_SESSION == "B":
    for seed in [1, 2]:
        run(["python", "complete_training.py",
             "--contrastive_ckpt", CONTRASTIVE_CKPT,
             "--output_dir", f"/kaggle/working/output_pct10_unconstrained_seed{seed}",
             "--labeled_percent", "10", "--epochs_seg", "30",
             "--seed", str(seed), "--unconstrained_pairwise",
             "--data_dir", "./input"], f"GATE U: unconstrained, seed {seed}")

In [ ]:
# Cell 10 [SESSION B]: GATE U-INIT - diagonal-init control, 3 seeds (~6h)
# Disambiguates 'remapping is a training attractor' vs 'the constraint just installs a good init'
if RUN_SESSION == "B":
    for seed in [0, 1, 2]:
        run(["python", "complete_training.py",
             "--contrastive_ckpt", CONTRASTIVE_CKPT,
             "--output_dir", f"/kaggle/working/output_pct10_uncdiag_seed{seed}",
             "--labeled_percent", "10", "--epochs_seg", "30",
             "--seed", str(seed), "--unconstrained_pairwise", "--unconstrained_diag_init",
             "--data_dir", "./input"], f"GATE U-INIT: diag-init control, seed {seed}")

In [ ]:
# Cell 11: Pairwise diagnostics on every new arm that exists (minutes; REAL DATA mandatory)
patterns = ["/kaggle/working/output_pct10_unconstrained_seed*/best_segmentation.pth",
            "/kaggle/working/output_pct10_uncdiag_seed*/best_segmentation.pth"]
ckpts = sorted(sum((glob.glob(p) for p in patterns), []))
print("Diagnosing:", *ckpts, sep="\n  ")
if ckpts:
    run(["python", "extract_pairwise_diagnostics.py", "--seg_ckpt", *ckpts,
         "--data_dir", "./input",
         "--output_dir", "/kaggle/working/diag_week1"], "PAIRWISE DIAGNOSTICS (universal protocol)")

In [ ]:
# Cell 12: k=3 eval of the NEW arms trained this session (eval-only, auto-detects head type)
new_arms = sorted(glob.glob("/kaggle/working/output_pct10_unconstrained_seed*/best_segmentation.pth")
                  + glob.glob("/kaggle/working/output_pct10_uncdiag_seed*/best_segmentation.pth"))
dumb = sorted(glob.glob("/kaggle/working/output_pct10_dumbpool_seed*/best_segmentation.pth"))
os.makedirs("/kaggle/working/gates", exist_ok=True)
if new_arms:
    run(["python", "measure_eval_noise.py", "--seg_ckpt", *new_arms, "--k", "3", "--both",
         "--output_json", "/kaggle/working/gates/removal_gap_new_arms.json"], "NEW ARMS: accuracy + removal gap")
if dumb:
    run(["python", "measure_eval_noise.py", "--seg_ckpt", *dumb, "--k", "3",
         "--output_json", "/kaggle/working/gates/dumbpool_eval.json"], "GATE D: dumb-pooling k=3 eval")

In [ ]:
# Cell 13: Collect + zip everything for download (run at END of each session)
results_dir = "/kaggle/working/week1_results"
os.makedirs(results_dir, exist_ok=True)
for d in glob.glob("/kaggle/working/output_pct10_*"):
    src = os.path.join(d, "best_segmentation.pth")
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(results_dir, os.path.basename(d) + "_best.pth"))
for d in ["diag_week1", "gates"]:
    if os.path.exists(f"/kaggle/working/{d}"):
        shutil.copytree(f"/kaggle/working/{d}", os.path.join(results_dir, d), dirs_exist_ok=True)
shutil.make_archive(f"/kaggle/working/week1_session{RUN_SESSION}_results", "zip", results_dir)
print(f"Download: /kaggle/working/week1_session{RUN_SESSION}_results.zip")
for root, _, files in os.walk(results_dir):
    for f in files:
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, results_dir):<60} {os.path.getsize(p)/1024:.1f} KB")